# chessgpt — Kaggle training run

Thin wrapper: clone the repo at a pinned commit, install it, find the dataset, run a short
GPU smoke test, then `python -m chessgpt.train`. All logic lives in the repo.

Pushed by `kaggle/run.sh`, which rewrites the **parameters** cell below. Outputs land in
`/kaggle/working/runs/<RUN_NAME>/` (`log.csv`, `best.pt`, `ckpt.pt`, `config.json`, `git_commit.txt`).
No secrets are needed: the repo is public and the dataset is attached.

In [ ]:
# Parameters: rewritten by kaggle/run.sh (kaggle/build_kernel.py)
REPO_URL = "https://github.com/OWNER/chessgpt.git"
GIT_REF = "main"
CONFIG = "configs/base.yaml"
RUN_NAME = "base"
MAX_HOURS = 11.0
SMOKE_TEST = True
RESUME = False


In [ ]:
import csv, glob, json, os, shutil, subprocess

REPO = "/tmp/chessgpt"  # outside /kaggle/working, so the code isn't saved as output


def sh(cmd: str) -> None:
    """Run a shell command, stream its output into the notebook, fail the run on error."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")


sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv")
shutil.rmtree(REPO, ignore_errors=True)
sh(f"git clone --quiet {REPO_URL} {REPO} && git -C {REPO} checkout --quiet {GIT_REF}")
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
print("commit", COMMIT)
sh(f"pip install --quiet -e {REPO}")


In [ ]:
# Locate the attached dataset (mount paths differ between Kaggle versions, so search).
metas = [m for m in glob.glob("/kaggle/input/**/meta.json", recursive=True)]
assert len(metas) == 1, f"expected exactly one meta.json under /kaggle/input, found {metas}"
INPUT_DIR = os.path.dirname(metas[0])

# Copy to the session's local disk. /kaggle/input is network-backed: training reads random
# 1 KB windows, and on the 1.25 GB train.bin those reads made training ~13x slower
# (2026-11-09). One sequential copy (~1 min) fixes it; the page cache then keeps it in RAM.
import time
DATA_DIR = "/tmp/data"
os.makedirs(DATA_DIR, exist_ok=True)
t = time.time()
for name in ("train.bin", "val.bin", "meta.json"):
    shutil.copy(os.path.join(INPUT_DIR, name), DATA_DIR)
size = sum(os.path.getsize(os.path.join(DATA_DIR, n)) for n in os.listdir(DATA_DIR))
print(f"copied {size / 1e9:.2f} GB from {INPUT_DIR} to {DATA_DIR} in {time.time() - t:.0f} s")

meta = json.load(open(os.path.join(DATA_DIR, "meta.json")))
print(DATA_DIR, "| vocab", meta["vocab_size"], "| tokens", meta["n_tokens"], "| games", meta["n_games"])


In [ ]:
# Smoke test: the real model config for 30 steps on the GPU, fp16 included.
# Catches GPU-only bugs (autocast, GradScaler) in a minute instead of hours into the run,
# and the timing gives the real tokens/s for planning.
if SMOKE_TEST:
    import time, yaml
    cfg = yaml.safe_load(open(f"{REPO}/{CONFIG}"))
    cfg["train"].update(max_iters=30, warmup_iters=5, eval_interval=10, eval_iters=2, ckpt_interval=30)
    yaml.safe_dump(cfg, open("/tmp/smoke.yaml", "w"))
    shutil.rmtree("/tmp/smoke", ignore_errors=True)
    t = time.time()
    sh(f"cd {REPO} && python -m chessgpt.train --config /tmp/smoke.yaml --data {DATA_DIR} --out /tmp/smoke")
    rows = list(csv.DictReader(open("/tmp/smoke/log.csv")))
    losses = [float(r["val_loss"]) for r in rows]
    assert all(l == l for l in losses), f"NaN loss in smoke test: {losses}"
    assert losses[-1] < losses[0], f"smoke test loss did not decrease: {losses}"
    tok_per_iter = cfg["train"]["batch_size"] * cfg["train"]["grad_accum_steps"] * cfg["model"]["block_size"]
    print(f"smoke OK: val {losses[0]:.3f} -> {losses[-1]:.3f}, "
          f"~{30 * tok_per_iter / (time.time() - t):,.0f} tokens/s (incl. evals and startup)")


In [ ]:
# The real run.
OUT = f"/kaggle/working/runs/{RUN_NAME}"
if RESUME:
    # Previous version of this notebook attached as a kernel source (run.sh --resume).
    prev = glob.glob(f"/kaggle/input/**/runs/{RUN_NAME}/ckpt.pt", recursive=True)
    assert len(prev) == 1, f"expected one previous ckpt.pt for {RUN_NAME}, found {prev}"
    shutil.copytree(os.path.dirname(prev[0]), OUT, dirs_exist_ok=True)
os.makedirs(OUT, exist_ok=True)
with open(f"{OUT}/git_commit.txt", "a") as f:
    f.write(COMMIT + "\n")
sh(f"cd {REPO} && python -m chessgpt.train --config {CONFIG} --data {DATA_DIR} --out {OUT} "
   f"--max-hours {MAX_HOURS}" + (" --resume" if RESUME else ""))


In [ ]:
for name in sorted(os.listdir(OUT)):
    print(f"{os.path.getsize(os.path.join(OUT, name)) / 1e6:9.1f} MB  {name}")
print(open(f"{OUT}/log.csv").read()[-1500:])
